![Cloud-First](../image/CloudFirst.png)

# SIT742: Modern Data Science
**(Module 05: Data Analytics)**

**LabClass M05A: Time Series Analytics With Australia To Hong Kong Data**

---

- Materials in this module have been developed to support practical learning in modern data science, big data processing, and applied analytics.
- The public notebook collection is available in [SIT742](https://github.com/tulip-lab/sit742).
- If you find an issue or bug in this document, please submit an issue at [SIT742](https://github.com/tulip-lab/sit742/issues).
- Audience: Honours and Master's students using the Deakin SIT742 practical and self-learning materials.

Prepared by the SIT742 Teaching Team.

Maintained through the [TULIP Lab](https://www.tulip.academy) FLIP workflow.

---

## LabClass M05A: Time Series Analytics With Australia To Hong Kong Data

<div align="center">

<table>
<thead>
<tr>
<th><strong>Item</strong></th>
<th><strong>Description</strong></th>
</tr>
</thead>
<tbody>
<tr>
<td align="left">Module context</td>
<td>This lab class practises forecasting, anomaly detection, and simple association-rule mining using monthly Australia to Hong Kong travel and search-interest data.</td>
</tr>
<tr>
<td align="left">Environment</td>
<td>Google Colab or local Jupyter with pandas, NumPy, matplotlib, scikit-learn, and statsmodels</td>
</tr>
<tr>
<td align="left">Main output</td>
<td>A forecast comparison, anomaly flags, and interpretable item-pair rules</td>
</tr>
<tr>
<td align="left">Related assessment</td>
<td>General practical skill development. Not directly assessed.</td>
</tr>
</tbody>
</table>

</div>

---


**Table of Contents**

- [1. Overview and Learning Goals](#m05a-overview)
- [2. Setup and Background](#m05a-setup)
- [3. Core Concepts](#m05a-core-concepts)
- [4. Guided Implementation](#m05a-guided-implementation)
- [5. Testing and Analysis](#m05a-testing)
- [6. Student Tasks](#m05a-student-tasks)
- [7. Reflection and References](#m05a-reflection)


<a id="m05a-overview"></a>

### 1. Overview and Learning Goals

This lab class uses `Australia_HK.csv` to practise time-series loading, moving averages, anomaly detection, ARIMA modelling, and simple association-rule reasoning.

By the end of this lab, students should be able to:

1. load and index a monthly time series;
2. compare moving-average and exponential smoothing baselines;
3. identify unusual observations with Isolation Forest;
4. fit and interpret an ARIMA model;
5. generate simple co-occurrence rules from high-interest search terms.


<a id="m05a-setup"></a>

### 2. Setup and Background

Use **Option A** in Google Colab or another online notebook environment. Use **Option B** only when the SIT742 repository is cloned locally.

The setup cell downloads `Australia_HK.csv` from the public SIT742 repository if a local copy is not available.


In [ ]:
# Optional, only if your runtime does not already provide these packages.
# !pip install pandas numpy matplotlib scikit-learn statsmodels

from collections import Counter
from itertools import combinations
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller

PUBLIC_DATA_BASE = "https://raw.githubusercontent.com/tulip-lab/sit742/develop/Jupyter/data"
DATA_FILE = "Australia_HK.csv"
DOWNLOAD_DIR = Path("data") / "m05_labclasses"
LOCAL_DATA_DIRS = [
    Path("../data"),
    Path("Jupyter/data"),
    Path("SIT742/Jupyter/data"),
]
ARIMA_ORDER = (1, 1, 1)

def resolve_or_download(filename):
    for data_dir in LOCAL_DATA_DIRS:
        candidate = data_dir / filename
        if candidate.exists():
            return candidate

    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    target = DOWNLOAD_DIR / filename
    if not target.exists():
        urlretrieve(f"{PUBLIC_DATA_BASE}/{filename}", target)
    return target

data_path = resolve_or_download(DATA_FILE)
print("Using data file:", data_path)


<a id="m05a-core-concepts"></a>

### 3. Core Concepts

The target series is monthly `arrival` counts. The remaining columns are search-interest indicators that can be converted into simple transactions for rule mining.

Key ideas:

- a time series must be sorted by time before modelling;
- moving averages and exponential smoothing provide simple baselines;
- anomaly detectors flag unusual patterns, not automatically wrong data;
- ARIMA requires careful interpretation of differencing and residuals;
- association rules should be treated as descriptive co-occurrence patterns.


<a id="m05a-guided-implementation"></a>

### 4. Guided Implementation

Load the data, set the date index, and inspect the `arrival` series.


In [ ]:
raw_df = pd.read_csv(data_path, parse_dates=["date"])
raw_df = raw_df.sort_values("date").set_index("date")
raw_df = raw_df.apply(pd.to_numeric, errors="coerce")

series = raw_df["arrival"].astype(float)
print(raw_df.shape)
display(raw_df.head())

ax = series.plot(figsize=(10, 4), title="Monthly arrivals")
ax.set_ylabel("Arrivals")
plt.tight_layout()
plt.show()


In [ ]:
def mean_absolute_percentage_error(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    non_zero = y_true != 0
    return np.mean(np.abs((y_true[non_zero] - y_pred[non_zero]) / y_true[non_zero])) * 100

moving_average = series.rolling(window=3, min_periods=1).mean()
residual = series - moving_average
mae = residual.abs().mean()
band = mae + residual.std()
upper_bound = moving_average + band
lower_bound = moving_average - band

plt.figure(figsize=(10, 4))
plt.plot(series.index, series, label="Observed")
plt.plot(moving_average.index, moving_average, label="3-month moving average")
plt.fill_between(series.index, lower_bound, upper_bound, alpha=0.2, label="MAE + 1 std band")
plt.legend()
plt.title("Moving-average baseline")
plt.tight_layout()
plt.show()

print("MAPE:", round(mean_absolute_percentage_error(series, moving_average), 2))


In [ ]:
isolation = IsolationForest(contamination=0.08, random_state=42)
anomaly_flag = isolation.fit_predict(series.to_frame(name="arrival"))
anomalies = series[anomaly_flag == -1]

plt.figure(figsize=(10, 4))
plt.plot(series.index, series, label="Observed")
plt.scatter(anomalies.index, anomalies, color="red", label="Isolation Forest anomaly")
plt.legend()
plt.title("Anomaly detection")
plt.tight_layout()
plt.show()

anomalies.to_frame("arrival")


In [ ]:
def exponential_smoothing(values, alpha=0.35):
    values = list(values)
    if not values:
        return []
    smoothed = [values[0]]
    for value in values[1:]:
        smoothed.append(alpha * value + (1 - alpha) * smoothed[-1])
    return pd.Series(smoothed, index=series.index)

exp_smooth = exponential_smoothing(series, alpha=0.35)

adf_stat, adf_pvalue, *_ = adfuller(series.dropna())
print("ADF statistic:", round(adf_stat, 3))
print("ADF p-value:", round(adf_pvalue, 4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(series.dropna(), lags=20, ax=axes[0])
plot_pacf(series.dropna(), lags=20, ax=axes[1], method="ywm")
plt.tight_layout()
plt.show()

model = ARIMA(series, order=ARIMA_ORDER)
fitted_model = model.fit()
fitted_values = fitted_model.fittedvalues

plt.figure(figsize=(10, 4))
plt.plot(series.index, series, label="Observed")
plt.plot(exp_smooth.index, exp_smooth, label="Exponential smoothing")
plt.plot(fitted_values.index, fitted_values, label=f"ARIMA{ARIMA_ORDER} fitted")
plt.legend()
plt.title("Forecasting baselines")
plt.tight_layout()
plt.show()

print(fitted_model.summary())


In [ ]:
interest_columns = [column for column in raw_df.columns if column != "arrival"]
selected_columns = interest_columns[:20]

def build_transactions(frame, columns, threshold=75):
    transactions = []
    for _, row in frame[columns].iterrows():
        items = [column for column in columns if pd.notna(row[column]) and row[column] >= threshold]
        if items:
            transactions.append(items)
    return transactions

def pair_rules(transactions, min_support=0.12, min_confidence=0.35):
    transaction_count = len(transactions)
    item_counts = Counter()
    pair_counts = Counter()
    for transaction in transactions:
        unique_items = sorted(set(transaction))
        item_counts.update(unique_items)
        pair_counts.update(combinations(unique_items, 2))

    rules = []
    for (left, right), pair_count in pair_counts.items():
        support = pair_count / transaction_count
        confidence_left = pair_count / item_counts[left]
        confidence_right = pair_count / item_counts[right]
        if support >= min_support and confidence_left >= min_confidence:
            rules.append((left, right, support, confidence_left))
        if support >= min_support and confidence_right >= min_confidence:
            rules.append((right, left, support, confidence_right))
    return sorted(rules, key=lambda rule: (rule[2], rule[3]), reverse=True)

transactions = build_transactions(raw_df, selected_columns, threshold=75)
rules = pair_rules(transactions)

print("Transactions:", len(transactions))
pd.DataFrame(rules[:10], columns=["if high interest in", "then high interest in", "support", "confidence"])


<a id="m05a-testing"></a>

### 5. Testing and Analysis

Run these checks before interpreting model outputs.


In [ ]:
assert len(series) == 84
assert series.index.is_monotonic_increasing
assert not series.isna().any()
assert len(anomalies) > 0
assert fitted_values.index.equals(series.index)
assert all(0 <= rule[2] <= 1 and 0 <= rule[3] <= 1 for rule in rules)

print("M05A checks passed.")


<a id="m05a-student-tasks"></a>

### 6. Student Tasks

<div align="center">

<table>
<thead>
<tr>
<th><strong>Task</strong></th>
<th><strong>What you need to do</strong></th>
<th><strong>Why it matters</strong></th>
<th><strong>Expected evidence</strong></th>
</tr>
</thead>
<tbody>
<tr>
<td align="left">Task 1</td>
<td>Change the moving-average window and compare MAPE.</td>
<td>Shows the trade-off between smoothing and responsiveness.</td>
<td>A metric comparison and one sentence of interpretation.</td>
</tr>
<tr>
<td align="left">Task 2</td>
<td>Adjust the Isolation Forest contamination value.</td>
<td>Shows how modelling assumptions affect anomaly flags.</td>
<td>A count of anomalies and a plot.</td>
</tr>
<tr>
<td align="left">Task 3</td>
<td>Use a different transaction threshold or column subset for rule mining.</td>
<td>Connects feature selection to rule interpretability.</td>
<td>A short rule table and explanation.</td>
</tr>
</tbody>
</table>

</div>


In [ ]:
# TODO: Complete Task 1 to Task 3 here.
# Keep a short written interpretation next to each output.


<a id="m05a-reflection"></a>

### 7. Reflection and References

Reflection questions:

1. Which output is descriptive and which output is predictive?
2. What assumptions does ARIMA make that may not hold for this series?
3. Why should association rules not be interpreted as causal relationships?

#### Further Readings

- statsmodels ARIMA documentation: https://www.statsmodels.org/stable/generated/statsmodels.tsa.arima.model.ARIMA.html
- scikit-learn Isolation Forest: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html
- pandas time series user guide: https://pandas.pydata.org/docs/user_guide/timeseries.html
